# Silver Layer - Olist E-commerce Dataset

**Objetivo:** Transformação e limpeza de dados da camada Bronze para a camada Silver seguindo a arquitetura Medalhão.

**Tabelas Criadas:**

**Dimensões (dim_):**
* `dim_consumidores` - Dados cadastrais de clientes únicos
* `dim_produtos` - Catálogo de produtos com atributos físicos
* `dim_vendedores` - Dados cadastrais de vendedores
* `dim_categoria_produtos_traducao` - Tradução de categorias PT→EN
* `dim_cotacao_dolar` - Cotação diária do dólar (calendário completo)

**Fatos (fat_):**
* `fat_pedidos` - Pedidos com métricas de entrega
* `fat_itens_pedidos` - Itens individuais de cada pedido
* `fat_pagamentos_pedidos` - Transações de pagamento
* `fat_avaliacoes_pedidos` - Avaliações dos clientes
* `fat_pedido_total` - Pedidos com valor total em BRL e USD

**Regras de Negócio Aplicadas:**
* Deduplicação de registros usando window functions
* Padronização de nomenclatura (português)
* Tradução de status e tipos de pagamento
* Cálculo de métricas derivadas (tempo de entrega, atrasos)
* Tratamento de valores nulos e datas inválidas
* Conversão cambial BRL → USD
* Preenchimento de cotações em finais de semana (forward fill)

## 1. Dimensão de Consumidores (dim_consumidores)

Tabela de dimensão contendo dados cadastrais únicos de clientes.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# lendo a tabela da camada bronze
df_bronze_customers = spark.read.format("delta").table("medalhao.bronze.tb_customers")

# definindo a window function 
window_spec = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

# 1. silver.dim_consumidores
df_dim_consumidores = (
    df_bronze_customers
    # adicionando uma coluna com o número da linha
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    # aplicando upper case onde necessário
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_name").alias("nome_consumidor"), 
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado")
    )
)

# salvando a tabela na silver
(df_dim_consumidores.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.dim_consumidores"))

print("Tabela silver.dim_consumidores criada!")
display(df_dim_consumidores.limit(5))

Tabela silver.dim_consumidores criada!


id_consumidor,prefixo_cep,nome_consumidor,cidade,estado
00012a2ce6f8dcda20d059ce98491703,6273,Dr. Davi Pinto,OSASCO,SP
000161a058600d5901f007fab4c27140,35550,Sr. Ravi Lucca Sousa,ITAPECERICA,MG
0001fd6190edaaf884bcaf3d49edf079,29830,Giovanna Ramos,NOVA VENECIA,ES
0002414f95344307404f0ace7a26f1d5,39664,Lívia Nogueira,MENDONCA,MG
000379cdec625522490c315e70c7a9fb,4841,Srta. Maria Laura Moura,SAO PAULO,SP


## 2. Fato de Pedidos (fat_pedidos)

Tabela fato contendo pedidos com métricas calculadas de tempo de entrega e indicadores de atraso.

In [0]:
# lendo a tabela da camada bronze
df_bronze_orders = spark.read.format("delta").table("medalhao.bronze.tb_orders")

# 2. silver.fat_pedidos
df_fat_pedidos = (
    df_bronze_orders
    # renomeando chaves principais
    .withColumnRenamed("order_id", "id_pedido")
    .withColumnRenamed("customer_id", "id_consumidor")
    .withColumnRenamed("order_purchase_timestamp", "data_pedido")
    
    # traduzindo os status 
    .withColumn("status",
        F.when(F.col("order_status") == "delivered", "entregue")
        .when(F.col("order_status") == "canceled", "cancelado")
        .when(F.col("order_status") == "shipped", "enviado")
        .when(F.col("order_status") == "processing", "processando")
        .when(F.col("order_status") == "invoiced", "faturado")
        .when(F.col("order_status") == "unavailable", "indisponível")
        .when(F.col("order_status") == "created", "criado")
        .when(F.col("order_status") == "approved", "aprovado")
        .otherwise(F.col("order_status"))
    )
    
    # criando colunas derivadas de data 
    .withColumn("tempo_entrega_dias", 
                F.datediff(F.col("order_delivered_customer_date"), F.col("data_pedido")))
    
    .withColumn("tempo_entrega_estimado_dias", 
                F.datediff(F.col("order_estimated_delivery_date"), F.col("data_pedido")))
    
    .withColumn("diferenca_entrega_dias", 
                F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
    
    # indicador textual de atraso
    .withColumn("entrega_no_prazo",
        F.when(F.col("status") != "entregue", "Não Entregue")
        .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Não")
    )
    
    # selecionando apenas as colunas que importam para a Silver
    .select(
        "id_pedido", "id_consumidor", "status", "data_pedido",
        "tempo_entrega_dias", "tempo_entrega_estimado_dias", 
        "diferenca_entrega_dias", "entrega_no_prazo"
    )
)

# salvando a tabela na silver
(df_fat_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.fat_pedidos"))

print("Tabela silver.fat_pedidos criada!")
display(df_fat_pedidos.limit(5))

Tabela silver.fat_pedidos criada!


id_pedido,id_consumidor,status,data_pedido,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,2017-10-02T10:56:33.000Z,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,2018-07-24T20:41:37.000Z,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,2018-08-08T08:38:49.000Z,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,2017-11-18T19:28:06.000Z,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,2018-02-13T21:18:39.000Z,3,13,-10,Sim


## 3. Fato de Itens de Pedidos (fat_itens_pedidos)

Tabela fato granular contendo cada item individual vendido em cada pedido com preços e frete.

In [0]:
# 3. silver.fat_itens_pedidos
df_bronze_items = spark.read.format("delta").table("medalhao.bronze.tb_order_items")

df_fat_itens_pedidos = (
    df_bronze_items.select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("price").alias("preco_BRL"),
        F.col("freight_value").alias("preco_frete")
    )
)

(df_fat_itens_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.fat_itens_pedidos"))

print("Tabela silver.fat_itens_pedidos criada!")
display(df_fat_itens_pedidos.limit(5))
print("-" * 50)

Tabela silver.fat_itens_pedidos criada!


id_pedido,id_item,id_produto,id_vendedor,preco_BRL,preco_frete
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.9,18.14


--------------------------------------------------


## 4. Fato de Pagamentos de Pedidos (fat_pagamentos_pedidos)

Tabela fato contendo transações de pagamento com tipos traduzidos e parcelas.

In [0]:
# 4. silver.fat_pagamentos_pedidos
df_bronze_payments = spark.read.format("delta").table("medalhao.bronze.tb_order_payments")

df_fat_pagamentos_pedidos = (
    df_bronze_payments
    .withColumn("tipo_pagamento",
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
        .when(F.col("payment_type") == "boleto", "Boleto")
        .when(F.col("payment_type") == "voucher", "Voucher")
        .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
        .when(F.col("payment_type") == "not_defined", "Não Definido")
        .otherwise(F.col("payment_type"))
    )
    # traduzindo as colunas
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),
        F.col("tipo_pagamento"),
        F.col("payment_installments").alias("parcelas_pagamento"),
        F.col("payment_value").alias("valor_pagamento")
    )
)

(df_fat_pagamentos_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.fat_pagamentos_pedidos"))

print("Tabela silver.fat_pagamentos_pedidos criada!")
display(df_fat_pagamentos_pedidos.limit(5))

Tabela silver.fat_pagamentos_pedidos criada!


id_pedido,sequencial_pagamento,tipo_pagamento,parcelas_pagamento,valor_pagamento
b81ef226f3fe1789b1e8b2acac839d17,1,Cartão de Crédito,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,Cartão de Crédito,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,Cartão de Crédito,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,Cartão de Crédito,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,Cartão de Crédito,2,128.45


## 5. Fato de Avaliações de Pedidos (fat_avaliacoes_pedidos)

Tabela fato contendo avaliações dos clientes com tratamento de valores nulos e datas inválidas.

In [0]:
# 5. silver.fat_avaliacoes_pedidos
df_bronze_reviews = spark.read.format("delta").table("medalhao.bronze.tb_order_reviews")

df_fat_avaliacoes_pedidos = (
    df_bronze_reviews
    # convertendo datas de forma segura e evitando que textos quebrem o código
    .withColumn("data_criacao_segura", F.try_to_timestamp(F.col("review_creation_date")))
    .withColumn("data_resposta_segura", F.try_to_timestamp(F.col("review_answer_timestamp")))
    
    # preenchendo comentários e títulos vazios
    .fillna("Sem comentário", subset=["review_comment_message"])
    .fillna("Sem título", subset=["review_comment_title"])
    
    # removendo pedidos inválidos e datas de comentário no futuro
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("data_criacao_segura") <= F.current_timestamp())
    
    # mapeamento e seleção final
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").cast("int").alias("nota_avaliacao"), 
        F.col("review_comment_title").alias("titulo_avaliacao"),   # adicionei o título, pois o PDF repetiu o mapeamento
        F.col("review_comment_message").alias("comentario_avaliacao"),
        F.col("data_criacao_segura").alias("data_criacao_avaliacao"),
        F.col("data_resposta_segura").alias("data_resposta_avaliacao")
    )
)

(df_fat_avaliacoes_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.fat_avaliacoes_pedidos"))

print("Tabela silver.fat_avaliacoes_pedidos criada!")
display(df_fat_avaliacoes_pedidos.limit(5))
print("-" * 50)

Tabela silver.fat_avaliacoes_pedidos criada!


id_avaliacao,id_pedido,nota_avaliacao,titulo_avaliacao,comentario_avaliacao,data_criacao_avaliacao,data_resposta_avaliacao
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,Sem título,Sem comentário,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,Sem título,Sem comentário,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,Sem título,Sem comentário,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Sem título,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Sem título,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z


--------------------------------------------------


## 6. Dimensão de Produtos (dim_produtos)

Tabela de dimensão contendo catálogo de produtos com atributos físicos e descritivos.

In [0]:
# 6. silver.dim_produtos
df_bronze_products = spark.read.format("delta").table("medalhao.bronze.tb_products")

# definindo a window function 
window_spec_prod = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_produtos = (
    df_bronze_products
    .withColumn("rn", F.row_number().over(window_spec_prod))
    .filter(F.col("rn") == 1)
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_name").alias("nome_produto"), 
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros"),
        F.col("product_photos_qty").alias("quantidade_fotos"),
        F.col("product_name_lenght").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").alias("tamanho_descricao_produto")
    )
)

(df_dim_produtos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.dim_produtos"))

print("Tabela silver.dim_produtos criada!")
display(df_dim_produtos.limit(5))
print("-" * 50)

Tabela silver.dim_produtos criada!


id_produto,nome_produto,categoria_produto,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,quantidade_fotos,tamanho_nome_produto,tamanho_descricao_produto
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,perfumaria,300.0,20.0,16.0,16.0,6.0,53.0,596.0
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,automotivo,1225.0,55.0,10.0,26.0,4.0,56.0,752.0
0009406fd7479715e4bef61dd91f2462,Toalha de Banho Premium,cama_mesa_banho,300.0,45.0,15.0,35.0,2.0,50.0,266.0
000b8f95fcb9e0096488278317764d19,Tábua de Corte,utilidades_domesticas,550.0,19.0,24.0,12.0,3.0,25.0,364.0
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,relogios_presentes,250.0,22.0,11.0,15.0,4.0,48.0,613.0


--------------------------------------------------


## 7. Dimensão de Vendedores (dim_vendedores)

Tabela de dimensão contendo dados cadastrais únicos de vendedores do marketplace.

In [0]:
# 7. silver.dim_vendedores
df_bronze_sellers = spark.read.format("delta").table("medalhao.bronze.tb_sellers")

# definindo a window function 
window_spec_vend = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_vendedores = (
    df_bronze_sellers
    .withColumn("rn", F.row_number().over(window_spec_vend))
    .filter(F.col("rn") == 1)
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_name").alias("nome_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado")
    )
)

(df_dim_vendedores.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.dim_vendedores"))

print("Tabela silver.dim_vendedores criada!")
display(df_dim_vendedores.limit(5))

Tabela silver.dim_vendedores criada!


id_vendedor,nome_vendedor,prefixo_cep,cidade,estado
0015a82c2db000af6aaaf3ae2ecb0532,Amanda Sá,9080,SANTO ANDRE,SP
001cca7ae9ae17fb1caed9dfb1094831,Vinicius Nogueira,29156,CARIACICA,ES
001e6ad469a905060d959994f1b41e4f,Ana Clara Moreira,24754,SAO GONCALO,RJ
002100f778ceb8431b7a1020ff7ab48f,Srta. Emanuella Rezende,14405,FRANCA,SP
003554e2dce176b5555353e4f3555ac8,Rebeca Costa,74565,GOIANIA,GO


## 8. Dimensão de Tradução de Categorias (dim_categoria_produtos_traducao)

Tabela de apoio contendo mapeamento de categorias de produtos em português e inglês.

In [0]:
# 8. silver.dim_categoria_produtos_traducao
df_bronze_translation = spark.read.format("delta").table("medalhao.bronze.tb_product_category_name_translation")

df_dim_categoria_produtos_traducao = (
    df_bronze_translation
    .select(
        F.col("product_category_name").alias("nome_produto_pt"), 
        F.col("product_category_name_english").alias("nome_produto_en")  
    )
)

(df_dim_categoria_produtos_traducao.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.dim_categoria_produtos_traducao"))

print("Tabela silver.dim_categoria_produtos_traducao criada!")
display(df_dim_categoria_produtos_traducao.limit(5))

Tabela silver.dim_categoria_produtos_traducao criada!


nome_produto_pt,nome_produto_en
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor


## 9. Dimensão de Cotação do Dólar + Fato de Pedido Total

Criação de calendário contínuo de cotações e tabela fato agregada com valores em BRL e USD.

In [0]:
# 9. silver.dim_cotacao_dolar
df_bronze_cotacao = spark.read.format("delta").table("medalhao.bronze.tb_cotacao_dolar")

df_cotacao_limpa = (
    df_bronze_cotacao
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .select("data_cotacao", F.col("cotacaoCompra").alias("cotacao_dolar"))
    .groupBy("data_cotacao").agg(F.last("cotacao_dolar").alias("cotacao_dolar"))
)

data_min_max = df_cotacao_limpa.agg(F.min("data_cotacao").alias("min"), F.max("data_cotacao").alias("max")).collect()[0]

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min_max['min']}'), to_date('{data_min_max['max']}'), interval 1 day)) as data_cotacao
""")

df_cotacao_completa = df_calendario.join(df_cotacao_limpa, "data_cotacao", "left")

window_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_dim_cotacao_dolar = (
    df_cotacao_completa
    .withColumn("cotacao_dolar_preenchida", F.last("cotacao_dolar", ignorenulls=True).over(window_ffill))
    .select("data_cotacao", F.col("cotacao_dolar_preenchida").alias("valor_cotacao"))
)

(df_dim_cotacao_dolar.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.dim_cotacao_dolar"))

print("Tabela silver.dim_cotacao_dolar criada!")
display(df_dim_cotacao_dolar.limit(5))
print("-" * 50)

# silver.fat_pedido_total
df_fat_pedidos = spark.read.format("delta").table("medalhao.silver.fat_pedidos")
df_fat_pagamentos = spark.read.format("delta").table("medalhao.silver.fat_pagamentos_pedidos")
df_dim_cotacao = spark.read.format("delta").table("medalhao.silver.dim_cotacao_dolar")

df_pagamentos_agrupados = (
    df_fat_pagamentos
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

df_fat_pedido_total = (
    df_fat_pedidos
    .join(df_pagamentos_agrupados, "id_pedido", "left")
    .withColumn("data_join_cotacao", F.to_date(F.col("data_pedido")))
    .join(df_dim_cotacao, F.col("data_join_cotacao") == df_dim_cotacao.data_cotacao, "left")
    
    .withColumn("valor_total_pago_brl", F.round(F.col("valor_total_pago_brl"), 2))
    .withColumn("valor_total_pago_usd", F.round(F.col("valor_total_pago_brl") / F.col("valor_cotacao"), 2))
    
    .select(
        "id_pedido", 
        "id_consumidor", 
        "status", 
        "valor_total_pago_brl", 
        "valor_total_pago_usd", 
        "data_pedido"
    )
)

(df_fat_pedido_total.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("medalhao.silver.fat_pedido_total"))

print("Tabela silver.fat_pedido_total criada!")
display(df_fat_pedido_total.limit(5))
print("-" * 50)

print("Começando otimização ZORDER...")
spark.sql("""
    OPTIMIZE medalhao.silver.fat_pedido_total
    ZORDER BY (id_pedido, data_pedido)
""")
print("Otimização concluída.")

Tabela silver.dim_cotacao_dolar criada!


data_cotacao,valor_cotacao
2016-09-01,3.2466
2016-09-02,3.2425
2016-09-03,3.2425
2016-09-04,3.2425
2016-09-05,3.2715


--------------------------------------------------
Tabela silver.fat_pedido_total criada!


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,38.71,12.24,2017-10-02T10:56:33.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,141.46,37.77,2018-07-24T20:41:37.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,179.12,47.75,2018-08-08T08:38:49.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,72.2,22.02,2017-11-18T19:28:06.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,28.62,8.72,2018-02-13T21:18:39.000Z


--------------------------------------------------
Começando otimização ZORDER...
Otimização concluída.
